# ATAC-RNA Model Comparison

This notebook:
1. Loads the lymphoma ATAC+RNA data and pre-trained topic models
2. Evaluates topic diversity and correlation with cell types for each hyperparameter configuration
3. Selects the best model based on cell type correlation
4. Trains MultiVI and MOFA+ models for comparison
5. Compares kNN (k=5) classification accuracy across all models

In [1]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Set paths
DATA_PATH = "/data/nelkazwi/share-topic/lymphoma_data/mdata_lymphoma.h5mu"
MODELS_DIR = Path("/data/omics_topic_models/atac_rna")
BASELINES_DIR = MODELS_DIR / "baselines"

## 1. Load Data

In [2]:
def load_data():
    """Load and preprocess ATAC+RNA lymphoma data (same as training script)."""
    mdata = mu.read_h5mu(DATA_PATH)

    # Binarize ATAC data
    mdata.mod['atac'].X.data = (mdata.mod['atac'].X.data > 0).astype(int)

    # Filter genes/regions with zero counts
    rna_total_counts = np.array(mdata.mod['rna'].X.sum(axis=0)).flatten()
    rna_nonzero_genes = rna_total_counts > 0
    mdata.mod['rna'] = mdata.mod['rna'][:, rna_nonzero_genes]

    atac_total_counts = np.array(mdata.mod['atac'].X.sum(axis=0)).flatten()
    atac_nonzero_regions = atac_total_counts > 0
    mdata.mod['atac'] = mdata.mod['atac'][:, atac_nonzero_regions]

    # Highly variable genes
    n_rna = min(2000, mdata.mod['rna'].n_vars)
    n_atac = min(20000, mdata.mod['atac'].n_vars)

    sc.pp.highly_variable_genes(mdata.mod['rna'], n_top_genes=n_rna, flavor='seurat_v3', subset=True)
    sc.pp.highly_variable_genes(mdata.mod['atac'], n_top_genes=n_atac, flavor='seurat_v3', subset=True)

    print(f"RNA: {mdata.mod['rna'].shape}")
    print(f"ATAC: {mdata.mod['atac'].shape}")

    return mdata

mdata = load_data()

# Get cell type labels
if 'cell_types' in mdata.mod['rna'].obs.columns:
    cell_types = mdata.mod['rna'].obs['cell_types'].values
    print(f"\nCell types found: {len(np.unique(cell_types))} unique types")
    print(cell_types.value_counts())
else:
    raise ValueError("Cell type annotations not found in data!")

RNA: (14566, 2000)
ATAC: (14566, 20000)

Cell types found: 15 unique types
 B/T mix             43
B                   503
Fibroblasts          27
Mono               1538
Mono/B mix          106
Mono/T mix          127
Stromal cells        82
T                  6876
T cycling          1503
Tumor B            2279
Tumor B cycling     632
Unkown               59
low GEX mix         693
pDC                  58
unknown mix          40
Name: count, dtype: int64


## 2. Load All Trained Models and Compute Metrics

In [3]:
from omics_topic.models import MultimodalAmortizedLDA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def parse_model_config(dirname):
    """Parse hyperparameters from directory name."""
    config = {}
    if 'horseshoe' in dirname:
        config['feature_prior_type'] = 'horseshoe'
    else:
        config['feature_prior_type'] = 'logistic_normal'
    
    if 'weight_cell' in dirname:
        config['weight_mode'] = 'cell'
    elif 'weight_universal' in dirname:
        config['weight_mode'] = 'universal'
    else:
        config['weight_mode'] = 'equal'
    
    if 'learnable_disp_global' in dirname:
        config['dispersion'] = 'learnable_global'
    elif 'learnable_disp_pergene' in dirname:
        config['dispersion'] = 'learnable_pergene'
    else:
        config['dispersion'] = 'fixed'
    
    return config

def compute_cell_type_correlation(theta, cell_types):
    """Compute ARI and NMI between topic assignments and cell type labels."""
    topic_assignments = np.argmax(theta, axis=1)
    le = LabelEncoder()
    cell_type_encoded = le.fit_transform(cell_types)
    ari = adjusted_rand_score(cell_type_encoded, topic_assignments)
    nmi = normalized_mutual_info_score(cell_type_encoded, topic_assignments)
    return {'ARI': ari, 'NMI': nmi}

def evaluate_knn_classification(X, y, k=5, test_size=0.2, random_state=42):
    """Evaluate kNN classification accuracy."""
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_val)
    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
    }

# List all model directories (exclude baselines)
model_dirs = sorted([d for d in MODELS_DIR.iterdir() if d.is_dir() and d.name.startswith("prior_")])
print(f"Found {len(model_dirs)} trained topic model directories")

In [4]:
import gc
import torch

# Load all models and compute metrics including kNN accuracy
results = []

# Need to setup data first for model loading
mdata_setup, modality_names, feat_counts = MultimodalAmortizedLDA.setup_mudata(
    mdata,
    modality_order=["rna", "atac"],
)
adata_flat = mdata.uns["_flattened_ann_data"]

cell_types_array = cell_types

for model_dir in model_dirs:
    model_path = model_dir / "model"
    if not model_path.exists():
        print(f"Skipping {model_dir.name}: no model found")
        continue
    
    print(f"Loading {model_dir.name}...")
    
    try:
        model = MultimodalAmortizedLDA.load(str(model_path), adata=adata_flat)
        theta = model.get_latent_representation(batch_size=mdata.n_obs)
        theta_array = theta.values.copy()  # Copy to numpy
        
        config = parse_model_config(model_dir.name)
        diversity = model.get_topic_diversity()
        diversity_rna = model.get_topic_diversity(modality='rna')
        diversity_atac = model.get_topic_diversity(modality='atac')
        corr_metrics = compute_cell_type_correlation(theta_array, cell_types)
        
        # kNN classification accuracy
        knn_metrics = evaluate_knn_classification(theta_array - 1, cell_types_array, k=5)
        
        result = {
            'model_name': model_dir.name,
            **config,
            'diversity': diversity,
            'diversity_rna': diversity_rna,
            'diversity_atac': diversity_atac,
            'ARI': corr_metrics['ARI'],
            'NMI': corr_metrics['NMI'],
            'knn_accuracy': knn_metrics['accuracy'],
            'knn_balanced_accuracy': knn_metrics['balanced_accuracy'],
            'theta': theta_array,
        }
        results.append(result)
        
        print(f"  Diversity: {diversity:.4f}, kNN Acc: {knn_metrics['accuracy']:.4f}, kNN Balanced: {knn_metrics['balanced_accuracy']:.4f}")
        
    except Exception as e:
        print(f"  Error loading model: {e}")
    
    finally:
        # Clear GPU memory after each model
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nSuccessfully loaded {len(results)} models")

In [5]:
# Create results DataFrame (without theta for display)
results_display = [{k: v for k, v in r.items() if k != 'theta'} for r in results]
df_results = pd.DataFrame(results_display)

# Sort by kNN balanced accuracy
_df_sorted = df_results.sort_values('knn_balanced_accuracy', ascending=False)
print("Model Comparison (sorted by kNN Balanced Accuracy):")
print(_df_sorted.to_string(index=False))

## 3. Select Best Model by kNN Accuracy

In [6]:
# Find best model by kNN balanced accuracy
best_idx = _df_sorted['knn_balanced_accuracy'].idxmax()
best_model_name = _df_sorted.loc[best_idx, 'model_name']
best_result = next(r for r in results if r['model_name'] == best_model_name)

print(f"Best model: {best_model_name}")
print(f"  Feature prior: {best_result['feature_prior_type']}")
print(f"  Weight mode: {best_result['weight_mode']}")
print(f"  Dispersion: {best_result['dispersion']}")
print(f"  kNN Accuracy: {best_result['knn_accuracy']:.4f}")
print(f"  kNN Balanced Accuracy: {best_result['knn_balanced_accuracy']:.4f}")
print(f"  NMI: {best_result['NMI']:.4f}")
print(f"  ARI: {best_result['ARI']:.4f}")
print(f"  Diversity: {best_result['diversity']:.4f}")

theta_best = best_result['theta']

In [7]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
df_pivot = _df_sorted.pivot_table(values='knn_balanced_accuracy', index='feature_prior_type', columns='weight_mode', aggfunc='mean')
sns.heatmap(df_pivot, annot=True, fmt='.3f', cmap='viridis', ax=ax)
ax.set_title('kNN Balanced Acc by Prior Type and Weight Mode')

ax = axes[1]
colors = _df_sorted['feature_prior_type'].map({'logistic_normal': 'blue', 'horseshoe': 'red'})
ax.scatter(_df_sorted['diversity'], _df_sorted['knn_balanced_accuracy'], c=colors, s=100, alpha=0.7)
ax.set_xlabel('Topic Diversity')
ax.set_ylabel('kNN Balanced Accuracy')
ax.set_title('Diversity vs kNN Balanced Accuracy')
ax.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Logistic Normal'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Horseshoe'),
])

ax = axes[2]
ax.scatter(_df_sorted['knn_accuracy'], _df_sorted['knn_balanced_accuracy'], c=colors, s=100, alpha=0.7)
ax.set_xlabel('kNN Accuracy')
ax.set_ylabel('kNN Balanced Accuracy')
ax.set_title('Accuracy vs Balanced Accuracy')

plt.tight_layout()
plt.show()

## 4. Load Baseline Models (MultiVI, MOFA+)

In [8]:
# Load pre-saved baseline latent representations
latent_multivi = None
multivi_path = BASELINES_DIR / "latent_multivi.npy"
if multivi_path.exists():
    latent_multivi = np.load(multivi_path)
    print(f"Loaded MultiVI latent: {latent_multivi.shape}")
else:
    print(f"MultiVI latent not found: {multivi_path}")

latent_mofa = None
mofa_path = BASELINES_DIR / "mdata_mofa.h5mu"
if mofa_path.exists():
    mdata_mofa = mu.read_h5mu(mofa_path)
    latent_mofa = np.asarray(mdata_mofa.obsm["X_mofa"])
    print(f"Loaded MOFA+ latent: {latent_mofa.shape}")
else:
    print(f"MOFA+ MuData not found: {mofa_path}")

latent_glue = None
glue_path = BASELINES_DIR / "latent_glue.npy"
if glue_path.exists():
    latent_glue = np.load(glue_path)
    print(f"Loaded GLUE latent: {latent_glue.shape}")
else:
    print(f"GLUE latent not found: {glue_path}")

## 5. kNN Classification Comparison (with error bars)

In [ ]:
# Multiple runs evaluation for error bars
n_runs = 5
seeds = [42, 123, 456, 789, 1011]

def evaluate_knn_multiple_runs(X, y, k=5, test_size=0.2, seeds=seeds):
    """Evaluate kNN classification with multiple random seeds."""
    accuracies = []
    balanced_accuracies = []
    for seed in seeds:
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=seed, stratify=y
        )
        knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
        knn.fit(X_train, y_train)
        y_pred = knn.predict(X_val)
        accuracies.append(accuracy_score(y_val, y_pred))
        balanced_accuracies.append(balanced_accuracy_score(y_val, y_pred))
    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "balanced_accuracy_mean": np.mean(balanced_accuracies),
        "balanced_accuracy_std": np.std(balanced_accuracies),
    }

In [ ]:
# Collect kNN results - only the best topic model + baselines
knn_results = {}

# Best topic model (already computed, but rerun with multiple seeds)
print("Evaluating Best Topic Model (5 runs)...")
knn_results["Omics-Topic"] = evaluate_knn_multiple_runs(theta_best - 1, cell_types_array, k=5)
print(f"  Accuracy: {knn_results['Omics-Topic']['accuracy_mean']:.4f} +/- {knn_results['Omics-Topic']['accuracy_std']:.4f}")
print(f"  Balanced Accuracy: {knn_results['Omics-Topic']['balanced_accuracy_mean']:.4f} +/- {knn_results['Omics-Topic']['balanced_accuracy_std']:.4f}")

if latent_multivi is not None:
    print("\nEvaluating MultiVI (5 runs)...")
    knn_results["MultiVI"] = evaluate_knn_multiple_runs(latent_multivi, cell_types_array, k=5)
    print(f"  Accuracy: {knn_results['MultiVI']['accuracy_mean']:.4f} +/- {knn_results['MultiVI']['accuracy_std']:.4f}")
    print(f"  Balanced Accuracy: {knn_results['MultiVI']['balanced_accuracy_mean']:.4f} +/- {knn_results['MultiVI']['balanced_accuracy_std']:.4f}")

if latent_mofa is not None:
    print("\nEvaluating MOFA+ (5 runs)...")
    knn_results["MOFA+"] = evaluate_knn_multiple_runs(latent_mofa, cell_types_array, k=5)
    print(f"  Accuracy: {knn_results['MOFA+']['accuracy_mean']:.4f} +/- {knn_results['MOFA+']['accuracy_std']:.4f}")
    print(f"  Balanced Accuracy: {knn_results['MOFA+']['balanced_accuracy_mean']:.4f} +/- {knn_results['MOFA+']['balanced_accuracy_std']:.4f}")

if latent_glue is not None:
    print("\nEvaluating GLUE (5 runs)...")
    knn_results["GLUE"] = evaluate_knn_multiple_runs(latent_glue, cell_types_array, k=5)
    print(f"  Accuracy: {knn_results['GLUE']['accuracy_mean']:.4f} +/- {knn_results['GLUE']['accuracy_std']:.4f}")
    print(f"  Balanced Accuracy: {knn_results['GLUE']['balanced_accuracy_mean']:.4f} +/- {knn_results['GLUE']['balanced_accuracy_std']:.4f}")

In [ ]:
summary_df = pd.DataFrame(
    {
        "Model": list(knn_results.keys()),
        "Accuracy": [r["accuracy_mean"] for r in knn_results.values()],
        "Accuracy_std": [r["accuracy_std"] for r in knn_results.values()],
        "Balanced Accuracy": [r["balanced_accuracy_mean"] for r in knn_results.values()],
        "Balanced Accuracy_std": [r["balanced_accuracy_std"] for r in knn_results.values()],
    }
).sort_values("Accuracy", ascending=False)

# Rename Omics-Topic to Ours
summary_df["Model"] = summary_df["Model"].replace("Omics-Topic", "Ours")

print("FINAL COMPARISON: kNN (k=5) Classification Results (mean +/- std over 5 runs)")
print(summary_df.to_string(index=False))

In [ ]:
# Side by side bar plot with error bars
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(summary_df))
width = 0.35

colors_acc = ["darkorange" if m == "Ours" else "steelblue" for m in summary_df["Model"]]
colors_bal = ["coral" if m == "Ours" else "lightsteelblue" for m in summary_df["Model"]]

bars1 = ax.bar(x - width/2, summary_df["Accuracy"], width, yerr=summary_df["Accuracy_std"], 
               label="Accuracy", color=colors_acc, alpha=0.85, capsize=3)
bars2 = ax.bar(x + width/2, summary_df["Balanced Accuracy"], width, yerr=summary_df["Balanced Accuracy_std"],
               label="Balanced Accuracy", color=colors_bal, alpha=0.85, capsize=3)

ax.set_ylabel("Accuracy")
ax.set_title("kNN (k=5) Cell Type Classification Comparison")
ax.set_xticks(x)
ax.set_xticklabels(summary_df["Model"], rotation=45, ha="right")
ax.legend()

for bar, value in zip(bars1, summary_df["Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
for bar, value in zip(bars2, summary_df["Balanced Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{value:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()

BASELINES_DIR.mkdir(parents=True, exist_ok=True)
summary_path = BASELINES_DIR / "knn_summary.csv"
summary_df.to_csv(summary_path, index=False)
topic_metrics_path = BASELINES_DIR / "topic_model_metrics.csv"
_df_sorted.to_csv(topic_metrics_path, index=False)
plot_path = BASELINES_DIR / "knn_comparison.png"
fig.savefig(plot_path, dpi=200)
plt.show()
print(f"\nSaved kNN summary to: {summary_path}")
print(f"Saved topic model metrics to: {topic_metrics_path}")
print(f"Saved comparison plot to: {plot_path}")

## 6. UMAP Visualizations

In [ ]:
import anndata as ad

def plot_umap_for_rep(X, labels, title_prefix, n_neighbors=15, min_dist=0.3):
    """Plot separate UMAP figures for Leiden and Cell Type."""
    adata = ad.AnnData(np.asarray(X))
    adata.obs["cell_type"] = pd.Categorical(labels)

    sc.pp.neighbors(adata, use_rep="X", n_neighbors=n_neighbors, metric="cosine")
    sc.tl.umap(adata, min_dist=min_dist)
    sc.tl.leiden(adata, key_added="leiden")

    # Plot 1: Leiden clustering
    fig1, ax1 = plt.subplots(figsize=(8, 6))
    sc.pl.umap(adata, color="leiden", ax=ax1, show=False, title=f"{title_prefix}: Leiden")
    plt.tight_layout()
    plt.show()

    # Plot 2: Cell type
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    sc.pl.umap(adata, color="cell_type", ax=ax2, show=False, title=f"{title_prefix}: Cell Type")
    plt.tight_layout()
    plt.show()


print("\nUMAPs for topic model (best)...")
plot_umap_for_rep(theta_best, cell_types_array, "Ours")

if latent_multivi is not None:
    print("\nUMAPs for MultiVI...")
    plot_umap_for_rep(latent_multivi, cell_types_array, "MultiVI")

if latent_mofa is not None:
    print("\nUMAPs for MOFA+...")
    plot_umap_for_rep(latent_mofa, cell_types_array, "MOFA+")

if latent_glue is not None:
    print("\nUMAPs for GLUE...")
    plot_umap_for_rep(latent_glue, cell_types_array, "GLUE")

## Summary

This notebook compared:
1. **Topic Models** with various hyperparameter combinations (prior type, weight mode, dispersion)
2. **MultiVI** - a deep generative model for multimodal single-cell data
3. **MOFA+** - Multi-Omics Factor Analysis
4. **GLUE** - Graph-Linked Unified Embedding (if available)

Key findings:
- Best topic model configuration selected based on kNN balanced accuracy
- kNN (k=5) classification accuracy compared across all models with error bars (5 runs)
- Results show relative performance of different integration approaches

In [ ]:
# Results already saved above
print("All results saved!")